# LLaVA-style medical captioner on ROCOv2 — BioMedCLIP → MLP → Qwen2.5-1.5B

The architecture from the LLaVA-Med precedent, sized for the 12 GB server:

```
image ─▶ BioMedCLIP ViT-B/16 (FROZEN) ─▶ 196 patch tokens (768-d, drop CLS)
                   │
        2-layer MLP + GELU (TRAINED) ─▶ 196 visual tokens in Qwen's space
                   │
   [ 196 visual tokens ; "a photo of" ; caption ] ─▶ Qwen2.5-1.5B (FROZEN + LoRA) ─▶ caption
```

**Why this, not the CLS+GPT-2 model:** that model gave the LLM a *single* visual
token and collapsed to ~12 template captions. LLaVA/LLaVA-Med feed the **grid of
196 patch tokens**, which is what breaks the bottleneck. We keep BioMedCLIP frozen,
train a small **MLP connector**, and LoRA-tune a **Qwen2.5-1.5B** decoder.

**Two-stage LLaVA-Med recipe** (both stages on the ROCO train split): **Stage 1**
trains only the MLP (aligns vision→Qwen); **Stage 2** adds LoRA on Qwen and
validates each epoch. bf16, batch 2 × accum 4. Measured peak ~7.8 GB / 12 GB, and
~62 min/epoch — the full run is an overnight job, so the headless
`train_roco_llava.py` in `tmux` is the intended way to run it; this notebook is the
readable/interactive copy.

In [ ]:
import os
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"   # ignore the stale HF token -> anonymous
import json, time, random
import torch
import torch.nn as nn
from PIL import Image

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.bfloat16   # fp16 -> AdamW-eps NaN; bf16 has fp32 range
print("device =", device, "| dtype =", dtype)

In [ ]:
# ── BioMedCLIP (frozen) + deterministic transform; we take the PATCH TOKENS ──
import open_clip
bmc, _pt, preprocess_val = open_clip.create_model_and_transforms(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
vision_encoder = bmc.visual.to(device).eval()
for p in vision_encoder.parameters():
    p.requires_grad = False
image_processor = preprocess_val

# Which ViT block the patch tokens come from. -2 (penultimate) is the LLaVA standard:
# BioMedCLIP is CLS-pooled (its pooled output IS the CLS token, verified exactly), so the
# LAST block is optimised solely to assemble the CLS and its patch tokens lose local detail.
VISION_LAYER = -2
_x = torch.randn(1, 3, 224, 224, device=device)
_t = (vision_encoder.trunk.forward_features(_x)[:, 1:, :] if VISION_LAYER == -1
      else vision_encoder.trunk.get_intermediate_layers(_x, n=abs(VISION_LAYER))[0])
print(f"patch tokens per image (layer {VISION_LAYER}):", tuple(_t.shape))

## The model

`visual_tokens` runs the frozen ViT, keeps the **196 patch tokens** (drops CLS),
and maps them with the MLP into Qwen's 1536-d space. `forward` prepends those 196
tokens to the tokenised `"a photo of" + caption`, masks the loss on the visual +
prompt positions, and lets Qwen compute the LM loss on the caption. `generate` does
the same but decodes. Trainable: the MLP (both stages) and Qwen LoRA (stage 2 only);
BioMedCLIP and the Qwen base are frozen.

In [ ]:
# ── Qwen2.5-1.5B decoder ─────────────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
QWEN = "Qwen/Qwen2.5-1.5B-Instruct"
qwen = AutoModelForCausalLM.from_pretrained(QWEN, dtype=dtype).to(device)   # ~3 GB, downloads once
tokenizer = AutoTokenizer.from_pretrained(QWEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Qwen hidden={qwen.config.hidden_size} layers={qwen.config.num_hidden_layers} vocab={qwen.config.vocab_size}")

In [ ]:
class LlavaBioMed(nn.Module):
    def __init__(self, ve, qwen):
        super().__init__()
        self.ve = ve
        H = qwen.config.hidden_size
        self.mlp = nn.Sequential(nn.Linear(768, H), nn.GELU(), nn.Linear(H, H))   # connector
        self.qwen = qwen
    def visual_tokens(self, images):
        with torch.no_grad():
            if VISION_LAYER == -1:
                feats = self.ve.trunk.forward_features(images)[:, 1:, :]   # last block, drop CLS
            else:
                # LLaVA standard (-2 = penultimate); CLS already stripped by timm
                feats = self.ve.trunk.get_intermediate_layers(images, n=abs(VISION_LAYER))[0]
        return self.mlp(feats.to(dtype))                              # [B,196,H]
    def forward(self, images, input_ids, attention_mask, labels):
        vis = self.visual_tokens(images)
        txt = self.qwen.get_input_embeddings()(input_ids)
        ie  = torch.cat([vis, txt], dim=1)
        att = torch.cat([torch.ones(vis.shape[:2], device=ie.device, dtype=attention_mask.dtype), attention_mask], 1)
        lab = torch.cat([torch.full(vis.shape[:2], -100, device=labels.device, dtype=labels.dtype), labels], 1)
        return self.qwen(inputs_embeds=ie, attention_mask=att, labels=lab)
    @torch.no_grad()
    def generate(self, images, prompt_ids, prompt_att, **gk):
        vis = self.visual_tokens(images)
        txt = self.qwen.get_input_embeddings()(prompt_ids)
        ie  = torch.cat([vis, txt], dim=1)
        att = torch.cat([torch.ones(vis.shape[:2], device=ie.device, dtype=prompt_att.dtype), prompt_att], 1)
        return self.qwen.generate(inputs_embeds=ie, attention_mask=att, **gk)

In [ ]:
model = LlavaBioMed(vision_encoder, qwen).to(device)
model.mlp = model.mlp.to(dtype)

def set_mode(train):
    # gradient checkpointing (train) vs KV cache (generation)
    if train:
        model.train(); model.qwen.config.use_cache = False
        model.qwen.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        model.eval(); model.qwen.gradient_checkpointing_disable(); model.qwen.config.use_cache = True

print("model built")

In [ ]:
# ── ROCOv2 captioning data ───────────────────────────────────────────────────
import pandas as pd
from torch.utils.data import Dataset, DataLoader

ROCO_DIR   = "/home/matei/rocov2"
CAP_PROMPT = "a photo of"
LLM_MAXLEN = 96
TRAIN_N    = 12000
VAL_LOSS_N = 1000
VAL_GEN_N  = 500
BATCH, ACCUM = 2, 4

def load_roco(split, n=None, seed=0):
    caps = pd.read_csv(os.path.join(ROCO_DIR, f"{split}_captions.csv")).dropna(subset=["Caption"]).reset_index(drop=True)
    d = os.path.join(ROCO_DIR, split)
    recs = [{"id": r.ID, "path": os.path.join(d, f"{r.ID}.jpg"), "caption": str(r.Caption)}
            for r in caps.itertuples() if os.path.isfile(os.path.join(d, f"{r.ID}.jpg"))]
    if n is not None:
        random.Random(seed).shuffle(recs); recs = recs[:n]
    return recs

train_recs    = load_roco("train", TRAIN_N)
val_loss_recs = load_roco("valid", VAL_LOSS_N)
val_gen_recs  = load_roco("valid", VAL_GEN_N)
print(f"train {len(train_recs)} | val-loss {len(val_loss_recs)} | val-gen {len(val_gen_recs)}")

PROMPT_LEN = len(tokenizer(CAP_PROMPT).input_ids)

class ROCOCaptionDataset(Dataset):
    def __init__(self, recs): self.recs = recs
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        image = image_processor(Image.open(r["path"]).convert("RGB"))
        full  = f"{CAP_PROMPT} {r['caption']}{tokenizer.eos_token}"
        t = tokenizer(full, padding="max_length", max_length=LLM_MAXLEN, truncation=True, return_tensors="pt")
        ids, att = t.input_ids.squeeze(0), t.attention_mask.squeeze(0)
        labels = ids.clone(); labels[att == 0] = -100; labels[:min(PROMPT_LEN, LLM_MAXLEN)] = -100
        return {"images": image, "input_ids": ids, "attention_mask": att, "labels": labels}

train_loader = DataLoader(ROCOCaptionDataset(train_recs), batch_size=BATCH, shuffle=True, num_workers=2)
val_loader   = DataLoader(ROCOCaptionDataset(val_loss_recs), batch_size=BATCH, shuffle=False, num_workers=2)

In [ ]:
# ── validation gen + metrics (same decoding + deberta BERTScore as all ROCO runs) ──
from tqdm.auto import tqdm
GEN_KWARGS = dict(max_new_tokens=40, min_new_tokens=8, num_beams=5, no_repeat_ngram_size=3,
                  length_penalty=1.0, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)

def caption_records(recs, batch_size=8):
    p = tokenizer(CAP_PROMPT, return_tensors="pt").to(device); preds = []
    for i in tqdm(range(0, len(recs), batch_size), desc="val gen", leave=False):
        chunk = recs[i:i+batch_size]
        pix = torch.stack([image_processor(Image.open(r["path"]).convert("RGB")) for r in chunk]).to(device)
        B = pix.shape[0]
        gen = model.generate(pix, p.input_ids.expand(B, -1), p.attention_mask.expand(B, -1), **GEN_KWARGS)
        preds.extend(s.strip() for s in tokenizer.batch_decode(gen, skip_special_tokens=True))
    return preds

@torch.no_grad()
def validation_loss(loader):
    set_mode(False); 
    tot = n = 0
    for b in tqdm(loader, desc="val loss", leave=False):
        out = model(b["images"].to(device), b["input_ids"].to(device), b["attention_mask"].to(device), b["labels"].to(device))
        k = b["images"].shape[0]; 
        tot += float(out.loss) * k; 
        n += k
    return tot / n

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from nltk.translate.meteor_score import meteor_score
import transformers.modeling_utils as _mu
_mu.check_torch_load_is_safe = lambda *a, **k: None
from bert_score import BERTScorer
_bert = BERTScorer(model_type="microsoft/deberta-xlarge-mnli", batch_size=8)
_bert._tokenizer.model_max_length = 512

def _norm(s): return " ".join(str(s).lower().split())
def compute_metrics(preds, refs):
    preds = [p if str(p).strip() else "." for p in preds]; 
    refs = [r if str(r).strip() else "." for r in refs]
    gts = {i:[_norm(refs[i])] for i in range(len(refs))}; 
    res = {i:[_norm(preds[i])] for i in range(len(preds))}
    bleu,_ = Bleu(4).compute_score(gts,res); 
    rouge,_ = Rouge().compute_score(gts,res); 
    cider,_ = Cider().compute_score(gts,res)
    meteor = sum(meteor_score([_norm(refs[i]).split()],_norm(preds[i]).split()) for i in range(len(preds)))/len(preds)
    _,_,F = _bert.score(preds, refs, batch_size=8, verbose=False)
    return {"BLEU-1":bleu[0],"BLEU-4":bleu[3],"METEOR":meteor,"ROUGE-L":rouge,"CIDEr":cider,"BERTScore-F1":F.mean().item()}

SAVE_DIR = "/home/matei/roco_llava_checkpoints"; 
os.makedirs(SAVE_DIR, exist_ok=True)
def save_ckpt(tag):
    lora = {k:v for k,v in model.qwen.state_dict().items() if "lora" in k.lower()}
    p = os.path.join(SAVE_DIR, f"roco_llava_{tag}.pt")
    torch.save({"mlp": model.mlp.state_dict(), "qwen_lora": lora, "vision_layer": VISION_LAYER}, p)
    return p

history = []; val_refs = [r["caption"] for r in val_gen_recs]

def train_epochs(optimizer, n_epochs, tag, do_val):
    from torch.nn.utils import clip_grad_norm_
    for epoch in range(n_epochs):
        set_mode(True); 
        optimizer.zero_grad(set_to_none=True); 
        running, t0 = 0.0, time.time()
        for step, b in enumerate(tqdm(train_loader, desc=f"{tag} epoch {epoch+1}/{n_epochs}")):
            out = model(b["images"].to(device), b["input_ids"].to(device), b["attention_mask"].to(device), b["labels"].to(device))
            (out.loss/ACCUM).backward()
            if (step+1) % ACCUM == 0:
                clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                optimizer.step(); 
                optimizer.zero_grad(set_to_none=True)
            running += out.loss.item()
        tl = running/len(train_loader)
        print(f"{tag} epoch {epoch+1} | {(time.time()-t0)/60:.1f} min | train {tl:.4f}")
        if not do_val: continue
        path = save_ckpt(f"{tag}_epoch{epoch+1}"); 
        vloss = validation_loss(val_loader); set_mode(False)
        vpreds = caption_records(val_gen_recs); vm = compute_metrics(vpreds, val_refs)
        history.append({"epoch":epoch+1,"stage":tag,"train_loss":tl,"val_loss":vloss,"path":path,**vm})
        print(f"  val {vloss:.4f} | BERTScore {vm['BERTScore-F1']:.4f} | CIDEr {vm['CIDEr']:.4f} | ROUGE-L {vm['ROUGE-L']:.4f}")
        print(f"  REF : {val_refs[0][:90]}\n  PRED: {vpreds[0][:90]}")
print("helpers ready")

In [ ]:
# ── STAGE 1: align the MLP connector (Qwen frozen) ───────────────────────────
from torch.optim import AdamW
for p in model.qwen.parameters():
    p.requires_grad = False
opt1 = AdamW(model.mlp.parameters(), lr=1e-3)
train_epochs(opt1, n_epochs=2, tag="s1", do_val=False)
save_ckpt("s1_final")
print("stage 1 done")

In [ ]:
# ── STAGE 2: inject LoRA on Qwen; train MLP + LoRA, validate each epoch ──────
from peft import LoraConfig, get_peft_model
model.qwen = get_peft_model(model.qwen, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM"))
lora_params = [p for p in model.qwen.parameters() if p.requires_grad]
print(f"stage-2 trainable: MLP {sum(p.numel() for p in model.mlp.parameters())/1e6:.1f}M + LoRA {sum(p.numel() for p in lora_params)/1e6:.1f}M")
opt2 = AdamW([{"params": model.mlp.parameters(), "lr": 2e-5}, {"params": lora_params, "lr": 2e-4}])
train_epochs(opt2, n_epochs=3, tag="s2", do_val=True)

In [ ]:
# ── Epoch selection on VALIDATION (test still untouched) ────────────────────
best = max(history, key=lambda r: r["BERTScore-F1"])
for h in history:
    print(f"  {h['stage']} epoch {h['epoch']} | val {h['val_loss']:.4f} | BERTScore {h['BERTScore-F1']:.4f} | CIDEr {h['CIDEr']:.4f}")
print(f"\nBEST: {best['stage']} epoch {best['epoch']}  ->  {best['path']}")
print("\nScore it ONCE on the full test set (same decoding + deberta BERTScore):")
print(f"  CKPT_PATH={best['path']} RUN_TAG=llava_finetuned \\")
print(f"    /home/matei/miniconda3/envs/vlm/bin/python /home/matei/caption_roco_llava.py")